<a href="https://colab.research.google.com/github/al7272-byte/Methane-Hydrate-ANN-PINN/blob/main/PINN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
import torch

# Setup GPU if available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/ML Grp Project Copy/Association_PINN/data cleaning.xlsx'
df = pd.read_excel(file_path)

In [ ]:
df.head()

Only Kinetics

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd


# Prepare the data
X = df[['Pressure (bar)', 'Bulk Temperature (Tb) (K)', 'Methane Mean Concentration (mol/m3)', 'Sub-cooling temperature(K)']]
y = df['Propogation Distance']

# PINN Setup (creates a grid of physical constraints)
pressure_physics = torch.linspace(30,80,30).view(-1,1).requires_grad_(True)
Temp_physics = torch.linspace(273,280,30).view(-1,1).requires_grad_(True)
Concn_CH4_physics = torch.linspace(746,2030,30).view(-1,1).requires_grad_(True)
Temp_sub_physics = torch.linspace(1,4,30).view(-1,1).requires_grad_(True)
x_physics = torch.cat([pressure_physics, Temp_physics], dim=1)
x_physics = torch.cat([x_physics, Concn_CH4_physics], dim=1)
x_physics = torch.cat([x_physics, Temp_sub_physics], dim=1)

# Split data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=50)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.9, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Normalize the target variable
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val = y_scaler.transform(y_val.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Constants based on the equation
h_prime = 3000
rho_H = 920
delta_H = 436.9
n = 2.5
delta_T = df['Delta T'].mean()
t = 60 # Sec

# Define the PINN model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

# Define the physics-informed loss function
def pinn_loss(y_true, y_pred, X, yhp):
    data_loss = nn.MSELoss()(y_pred, y_true.view(-1, 1)) # Difference between prediction and actual

    dx  = torch.autograd.grad(yhp, x_physics, torch.ones_like(yhp), create_graph=True)[0]


    # Physics-informed loss component
    physics_pred = dx[:, 0] - (t * h_prime * (delta_T)**n) / (rho_H * delta_H)
    physics_pred_tensor = torch.tensor(physics_pred, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss = nn.MSELoss()(yhp, physics_pred_tensor)


    # Combine losses with adjusted balancing factor
    total_loss = data_loss + (10**-2) * physics_loss
    #print(f"Data Loss: {data_loss.item()}, Physics Loss: {physics_loss.item()}, Total Loss: {total_loss.item()}")
    return total_loss

# Training configuration
n_epochs = 8000
learning_rate = 0.001

# Initialize model, optimizer, and scheduler
model = PINN()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)  # Learning rate decay

# Training loop without early stopping, L2 regularization, or gradient clipping
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    yh_physics = model(x_physics)
    loss = pinn_loss(y_train_tensor, y_pred, X_train_tensor,yh_physics)
    loss.backward()

    optimizer.step()
    scheduler.step()  # Update learning rate

# Evaluate on test set
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    y_test_pred_inv = y_scaler.inverse_transform(y_test_pred.numpy())
    y_test_inv = y_scaler.inverse_transform(y_test_tensor.numpy())

# Calculate error metrics
mse = mean_squared_error(y_test_inv, y_test_pred_inv)
mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
r2 = r2_score(y_test_inv, y_test_pred_inv)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-Squared (R2 Score): {r2:.4f}")

# Scatter plot: Actual vs Predicted Values
plt.figure(figsize=(10, 8))
plt.scatter(y_test_inv, y_test_pred_inv, color='dodgerblue', alpha=0.7, edgecolor='black', s=100, label='Predicted vs Actual')
plt.plot([min(y_test_inv), max(y_test_inv)], [min(y_test_inv), max(y_test_inv)], color='red', linestyle='--', linewidth=2, label='Perfect Fit Line')

# Customizing the plot
plt.xlabel('Actual Values', fontsize=14, fontweight='bold')
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold')
plt.title('Actual vs Predicted Values (Test Set)', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Enhance ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('scatter_actual_vs_predicted.png', dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()

# Line plot: Comparison of Actual and Predicted Values
plt.figure(figsize=(14, 8))
plt.plot(y_test_inv, label='Actual', color='blue', marker='o', markersize=6, linewidth=2, alpha=0.8)
plt.plot(y_test_pred_inv, label='Predicted', color='orange', marker='x', markersize=8, linewidth=2, linestyle='--', alpha=0.8)

# Customizing the plot
plt.xlabel('Sample Index', fontsize=14, fontweight='bold')
plt.ylabel('Propagation Rate', fontsize=14, fontweight='bold')
plt.title('Comparison of Actual and Predicted Values', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Enhance ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('comparison_actual_predicted.png', dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()


In [ ]:
weights = model.fc1.weight.data.cpu().numpy()
importance = np.mean(np.abs(weights), axis=0)
print("Feature importance scores (first layer):", importance)


In [ ]:
X_train

Combined Kinetics + heat transfer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Prepare the data
X = df[['Pressure (bar)', 'Bulk Temperature (Tb) (K)', 'Methane Mean Concentration (mol/m3)', 'Sub-cooling temperature(K)']]
y = df['Propogation Distance']

pressure_physics = torch.linspace(30,80,30).view(-1,1).requires_grad_(True)
Temp_physics = torch.linspace(273,280,30).view(-1,1).requires_grad_(True)
Concn_CH4_physics = torch.linspace(746,2030,30).view(-1,1).requires_grad_(True)
Temp_sub_physics = torch.linspace(1,4,30).view(-1,1).requires_grad_(True)
x_physics = torch.cat([pressure_physics, Temp_physics], dim=1)
x_physics = torch.cat([x_physics, Concn_CH4_physics], dim=1)
x_physics = torch.cat([x_physics, Temp_sub_physics], dim=1)

# Split data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=50)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.9, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Normalize the target variable
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val = y_scaler.transform(y_val.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Constants based on the equation
h_prime = 3000
rho_H = 920
delta_H = 436.9
n = 2.5
delta_T = df['Delta T'].mean()
t = 60 # Sec

# Kinetics

k1 = 2.71 * 10**-3 #m/s
kc = 7.305 * 10**-3 #m/s
M_CH4 = 0.016 #kg/mol
W_CH4 = 0.13
row_H = 0.083 #kg/m3
Cm = 1.38 * 10**3 #mol/m3

# Define the PINN model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

# Define the physics-informed loss function
def pinn_loss(y_true, y_pred, X, yhp):
    data_loss = nn.MSELoss()(y_pred, y_true.view(-1, 1))

    dx  = torch.autograd.grad(yhp, x_physics, torch.ones_like(yhp), create_graph=True)[0]


    # Physics-informed loss component for Heat transfers
    physics_pred_heat = dx[:, 0] - (t * h_prime * (delta_T)**n) / (rho_H * delta_H)
    physics_pred_tensor_heat = torch.tensor(physics_pred_heat, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss_heat = nn.MSELoss()(yhp, physics_pred_tensor_heat)

    # Physics-informed loss component for kinetics
    physics_pred_kinetics = dx[:, 0] - (t*k1*kc/(k1 + kc))*(M_CH4*Cm/(W_CH4*row_H))
    physics_pred_tensor_kinetics = torch.tensor(physics_pred_kinetics, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss_kinetics = nn.MSELoss()(yhp, physics_pred_tensor_kinetics)


    # Combine losses with adjusted balancing factor
    total_loss = data_loss + (10**-1) * physics_loss_heat + (10**-1) * physics_loss_kinetics
    #print(f"Data Loss: {data_loss.item()}, Physics Loss Heat: {physics_loss_heat.item()}, Physics Loss Kinetics: {physics_loss_kinetics.item()}, Total Loss: {total_loss.item()}")
    return total_loss

# Training configuration
n_epochs = 8000
learning_rate = 0.01

# Initialize model, optimizer, and scheduler
model = PINN()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)  # Learning rate decay

# Training loop without early stopping, L2 regularization, or gradient clipping
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    yh_physics = model(x_physics)
    loss = pinn_loss(y_train_tensor, y_pred, X_train_tensor,yh_physics)
    loss.backward()

    optimizer.step()
    scheduler.step()  # Update learning rate

# Evaluate on test set
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    y_test_pred_inv = y_scaler.inverse_transform(y_test_pred.numpy())
    y_test_inv = y_scaler.inverse_transform(y_test_tensor.numpy())

# Calculate error metrics
mse = mean_squared_error(y_test_inv, y_test_pred_inv)
mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
r2 = r2_score(y_test_inv, y_test_pred_inv)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-Squared (R2 Score): {r2:.4f}")

# Scatter Plot: Predicted vs Actual with Perfect Fit Line
plt.figure(figsize=(10, 8))
plt.scatter(y_test_inv, y_test_pred_inv, color='dodgerblue', edgecolor='black', s=80, alpha=0.7, label='Predicted vs Actual')
plt.plot([min(y_test_inv), max(y_test_inv)], [min(y_test_inv), max(y_test_inv)], color='red', linestyle='--', linewidth=2, label='Perfect Fit Line')

# Customize plot
plt.xlabel('Actual Values', fontsize=14, fontweight='bold')
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold')
plt.title('Actual vs Predicted Values (Test Set)', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Adjust ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('actual_vs_predicted_scatter_improved.png', dpi=300, bbox_inches='tight')

# Show plot
plt.tight_layout()
plt.show()

# Line Plot: Comparison of Actual and Predicted Values
plt.figure(figsize=(14, 8))
plt.plot(y_test_inv, label='Actual', color='blue', marker='o', markersize=6, linewidth=2, linestyle='-', alpha=0.8)
plt.plot(y_test_pred_inv, label='Predicted', color='orange', marker='x', markersize=6, linewidth=2, linestyle='--', alpha=0.8)

# Customize plot
plt.xlabel('Sample Index', fontsize=14, fontweight='bold')
plt.ylabel('Propagation Rate', fontsize=14, fontweight='bold')
plt.title('Comparison of Actual and Predicted Values', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='upper left')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Adjust ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('actual_vs_predicted_comparison_improved.png', dpi=300, bbox_inches='tight')

# Show plot
plt.tight_layout()
plt.show()



Only Heat Transfer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Prepare the data
X = df[['Pressure (bar)', 'Bulk Temperature (Tb) (K)', 'Methane Mean Concentration (mol/m3)', 'Sub-cooling temperature(K)']]
y = df['Propogation Distance']

pressure_physics = torch.linspace(30,80,30).view(-1,1).requires_grad_(True)
Temp_physics = torch.linspace(273,280,30).view(-1,1).requires_grad_(True)
Concn_CH4_physics = torch.linspace(746,2030,30).view(-1,1).requires_grad_(True)
Temp_sub_physics = torch.linspace(1,4,30).view(-1,1).requires_grad_(True)
x_physics = torch.cat([pressure_physics, Temp_physics], dim=1)
x_physics = torch.cat([x_physics, Concn_CH4_physics], dim=1)
x_physics = torch.cat([x_physics, Temp_sub_physics], dim=1)

# Split data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=50)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.9, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Normalize the target variable
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val = y_scaler.transform(y_val.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Constants based on the equation
h_prime = 3000
rho_H = 920
delta_H = 436.9
n = 2.5
delta_T = df['Delta T'].mean()
t = 60 # Sec

# Kinetics

k1 = 2.71 * 10**-3 #m/s
kc = 7.305 * 10**-3 #m/s
M_CH4 = 0.016 #kg/mol
W_CH4 = 0.13
row_H = 0.083 #kg/m3
Cm = 1.38 * 10**3 #mol/m3

# Define the PINN model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

# Define the physics-informed loss function
def pinn_loss(y_true, y_pred, X, yhp):
    data_loss = nn.MSELoss()(y_pred, y_true.view(-1, 1))

    dx  = torch.autograd.grad(yhp, x_physics, torch.ones_like(yhp), create_graph=True)[0]


    # Physics-informed loss component for Heat transfers
    physics_pred_heat = dx[:, 0] - (t * h_prime * (delta_T)**n) / (rho_H * delta_H)
    physics_pred_tensor_heat = torch.tensor(physics_pred_heat, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss_heat = nn.MSELoss()(yhp, physics_pred_tensor_heat)


    # Combine losses with adjusted balancing factor
    total_loss = data_loss + (10**-1) * physics_loss_heat
    #print(f"Data Loss: {data_loss.item()}, Physics Loss Heat: {physics_loss_heat.item()}, Physics Loss Kinetics: {physics_loss_kinetics.item()}, Total Loss: {total_loss.item()}")
    return total_loss

# Training configuration
n_epochs = 8000
learning_rate = 0.01

# Initialize model, optimizer, and scheduler
model = PINN()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)  # Learning rate decay

# Training loop without early stopping, L2 regularization, or gradient clipping
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    yh_physics = model(x_physics)
    loss = pinn_loss(y_train_tensor, y_pred, X_train_tensor,yh_physics)
    loss.backward()

    optimizer.step()
    scheduler.step()  # Update learning rate

# Evaluate on test set
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    y_test_pred_inv = y_scaler.inverse_transform(y_test_pred.numpy())
    y_test_inv = y_scaler.inverse_transform(y_test_tensor.numpy())

# Calculate error metrics
mse = mean_squared_error(y_test_inv, y_test_pred_inv)
mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
r2 = r2_score(y_test_inv, y_test_pred_inv)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-Squared (R2 Score): {r2:.4f}")

# Scatter Plot: Predicted vs Actual with Perfect Fit Line
plt.figure(figsize=(10, 8))
plt.scatter(y_test_inv, y_test_pred_inv, color='dodgerblue', edgecolor='black', s=80, alpha=0.7, label='Predicted vs Actual')
plt.plot([min(y_test_inv), max(y_test_inv)], [min(y_test_inv), max(y_test_inv)], color='red', linestyle='--', linewidth=2, label='Perfect Fit Line')

# Customize plot
plt.xlabel('Actual Values', fontsize=14, fontweight='bold')
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold')
plt.title('Actual vs Predicted Values (Test Set)', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Adjust ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('actual_vs_predicted_scatter_improved.png', dpi=300, bbox_inches='tight')

# Show plot
plt.tight_layout()
plt.show()

# Line Plot: Comparison of Actual and Predicted Values
plt.figure(figsize=(14, 8))
plt.plot(y_test_inv, label='Actual', color='blue', marker='o', markersize=6, linewidth=2, linestyle='-', alpha=0.8)
plt.plot(y_test_pred_inv, label='Predicted', color='orange', marker='x', markersize=6, linewidth=2, linestyle='--', alpha=0.8)

# Customize plot
plt.xlabel('Sample Index', fontsize=14, fontweight='bold')
plt.ylabel('Propagation Rate', fontsize=14, fontweight='bold')
plt.title('Comparison of Actual and Predicted Values', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='upper left')
plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Adjust ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Save the plot (optional)
# plt.savefig('actual_vs_predicted_comparison_improved.png', dpi=300, bbox_inches='tight')

# Show plot
plt.tight_layout()
plt.show()



In [ ]:
# Enhanced PINN model with improved visualizations
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator

# Set the visual style for all plots
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

# Define a custom color palette
custom_palette = sns.color_palette("viridis", 4)
sns.set_palette(custom_palette)

# Assuming df is loaded as in your original code
# Prepare the data
X = df[['Pressure (bar)', 'Bulk Temperature (Tb) (K)', 'Methane Mean Concentration (mol/m3)', 'Sub-cooling temperature(K)']]
y = df['Propogation Distance']

pressure_physics = torch.linspace(30, 80, 30).view(-1, 1).requires_grad_(True)
Temp_physics = torch.linspace(273, 280, 30).view(-1, 1).requires_grad_(True)
Concn_CH4_physics = torch.linspace(746, 2030, 30).view(-1, 1).requires_grad_(True)
Temp_sub_physics = torch.linspace(1, 4, 30).view(-1, 1).requires_grad_(True)
x_physics = torch.cat([pressure_physics, Temp_physics], dim=1)
x_physics = torch.cat([x_physics, Concn_CH4_physics], dim=1)
x_physics = torch.cat([x_physics, Temp_sub_physics], dim=1)

# Split data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=50)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.9, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Normalize the target variable
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val = y_scaler.transform(y_val.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Constants based on the equation
h_prime = 3000
rho_H = 920
delta_H = 436.9
n = 2.5
delta_T = df['Delta T'].mean()
t = 60  # Sec

# Kinetics constants
k1 = 2.71 * 10**-3  # m/s
kc = 7.305 * 10**-3  # m/s
M_CH4 = 0.016  # kg/mol
W_CH4 = 0.13
row_H = 0.083  # kg/m3
Cm = 1.38 * 10**3  # mol/m3

# Define the PINN model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 1)

        # Apply proper initialization for better convergence
        nn.init.xavier_normal_(self.fc1.weight)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.xavier_normal_(self.fc4.weight)
        nn.init.xavier_normal_(self.fc5.weight)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

# Define the physics-informed loss function
def pinn_loss(y_true, y_pred, X, yhp):
    data_loss = nn.MSELoss()(y_pred, y_true.view(-1, 1))

    dx = torch.autograd.grad(yhp, x_physics, torch.ones_like(yhp), create_graph=True)[0]

    # Physics-informed loss component for Heat transfers
    physics_pred_heat = dx[:, 0] - (t * h_prime * (delta_T)**n) / (rho_H * delta_H)
    physics_pred_tensor_heat = torch.tensor(physics_pred_heat, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss_heat = nn.MSELoss()(yhp, physics_pred_tensor_heat)

    # Physics-informed loss component for kinetics
    physics_pred_kinetics = dx[:, 0] - (t*k1*kc/(k1 + kc))*(M_CH4*Cm/(W_CH4*row_H))
    physics_pred_tensor_kinetics = torch.tensor(physics_pred_kinetics, dtype=torch.float32, device=y_pred.device).reshape(-1, 1)

    # Physics loss
    physics_loss_kinetics = nn.MSELoss()(yhp, physics_pred_tensor_kinetics)

    # Combine losses with adjusted balancing factor
    total_loss = data_loss + (10**-1) * physics_loss_heat + (10**-1) * physics_loss_kinetics

    return total_loss

# Training configuration
n_epochs = 8000
learning_rate = 0.01

# Initialize model, optimizer, and scheduler
model = PINN()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)  # Learning rate decay

# Lists to store loss values for plotting
train_losses = []

# Training loop with loss tracking
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    yh_physics = model(x_physics)
    loss = pinn_loss(y_train_tensor, y_pred, X_train_tensor, yh_physics)
    loss.backward()

    optimizer.step()
    scheduler.step()  # Update learning rate

    # Store loss for plotting
    train_losses.append(loss.item())

    # Print progress every 500 epochs
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.6f}")

# Plot training loss curve
plt.figure(figsize=(12, 6))
plt.plot(train_losses, color=custom_palette[0], linewidth=2)
plt.title('Training Loss over Epochs', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.yscale('log')  # Log scale often helps visualize loss curves better
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('pinn_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

# Evaluate on test set
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    y_test_pred_inv = y_scaler.inverse_transform(y_test_pred.numpy())
    y_test_inv = y_scaler.inverse_transform(y_test_tensor.numpy())

# Calculate error metrics
mse = mean_squared_error(y_test_inv, y_test_pred_inv)
mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
r2 = r2_score(y_test_inv, y_test_pred_inv)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-Squared (R2 Score): {r2:.4f}")

# Store metrics in a dictionary for plotting
metrics = {
    'mse': mse,
    'mae': mae,
    'r2': r2
}

# 1. Enhanced Scatter Plot
def plot_scatter_comparison(y_true, y_pred, metrics, title="Actual vs Predicted Values"):
    fig, ax = plt.subplots(figsize=(12, 9), dpi=100)

    # Calculate error for coloring
    errors = np.abs(y_true.flatten() - y_pred.flatten())

    # Create scatter plot with colormap based on error
    scatter = ax.scatter(y_true, y_pred,
                      c=errors,
                      cmap='viridis',
                      s=100,
                      alpha=0.8,
                      edgecolor='white',
                      linewidth=0.8)

    # Add perfect prediction line
    line_min = min(y_true.min(), y_pred.min()) * 0.9
    line_max = max(y_true.max(), y_pred.max()) * 1.1
    ax.plot([line_min, line_max], [line_min, line_max],
            color='firebrick',
            linestyle='--',
            linewidth=2.5,
            label='Perfect Prediction')

    # Add colorbar to show error magnitude
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
    cbar.set_label('Absolute Error', rotation=270, labelpad=20, fontsize=12)

    # Annotations and styling
    ax.set_xlabel('Actual Values', fontweight='bold')
    ax.set_ylabel('Predicted Values', fontweight='bold')

    # Set equal scales
    ax.set_aspect('equal', adjustable='box')

    # Expand limits slightly
    margin = 0.1 * (line_max - line_min)
    ax.set_xlim(line_min - margin, line_max + margin)
    ax.set_ylim(line_min - margin, line_max + margin)

    # Create metrics text box
    metrics_text = (f"MSE: {metrics['mse']:.4f}\n"
                    f"MAE: {metrics['mae']:.4f}\n"
                    f"R²: {metrics['r2']:.4f}")

    # Add metrics text box
    props = dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='gray')
    ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props)

    # Title with enhanced styling
    ax.set_title(title, fontweight='bold', pad=20)

    # Set grid style
    ax.grid(True, linestyle='--', alpha=0.7)

    # Legend
    ax.legend(loc='lower right', frameon=True, framealpha=0.9)

    # Tight layout
    plt.tight_layout()

    return fig

# 2. Enhanced Line Plot
def plot_line_comparison(y_true, y_pred, title="Comparison of Actual and Predicted Values"):
    fig, ax = plt.subplots(figsize=(14, 8), dpi=100)

    # Create indices
    indices = np.arange(len(y_true))

    # Error bands
    error = np.abs(y_true.flatten() - y_pred.flatten())

    # Plot actual values
    ax.plot(indices, y_true,
            marker='o',
            markersize=8,
            linewidth=2.5,
            label='Actual',
            color=custom_palette[0],
            alpha=0.9)

    # Plot predicted values
    ax.plot(indices, y_pred,
            marker='x',
            markersize=10,
            linewidth=2.5,
            linestyle='--',
            label='Predicted',
            color=custom_palette[1],
            alpha=0.9)

    # Add error regions
    ax.fill_between(indices,
                    y_pred.flatten() - error/2,
                    y_pred.flatten() + error/2,
                    alpha=0.2,
                    color=custom_palette[1],
                    label='Error Range')

    # Set integer x-ticks
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Styling enhancements
    ax.set_xlabel('Sample Index', fontweight='bold')
    ax.set_ylabel('Propagation Distance', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=20)

    # Add a horizontal line for the mean of actual values
    ax.axhline(y=np.mean(y_true),
               color='green',
               linestyle=':',
               linewidth=1.5,
               label='Mean Actual Value',
               alpha=0.7)

    # Set grid style
    ax.grid(True, linestyle='--', alpha=0.5)

    # Legend with styling
    ax.legend(loc='upper right',
              frameon=True,
              framealpha=0.95,
              edgecolor='gray')

    # Add annotations for extreme points
    max_error_idx = np.argmax(error)
    ax.annotate(f'Max Error: {error[max_error_idx]:.2f}',
                xy=(indices[max_error_idx], y_pred.flatten()[max_error_idx]),
                xytext=(indices[max_error_idx]+2, y_pred.flatten()[max_error_idx]+0.2),
                arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
                fontsize=11,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

    # Tight layout
    plt.tight_layout()

    return fig

# 3. Residual analysis plot
def plot_residuals(y_true, y_pred, title="Residual Analysis"):
    residuals = y_pred.flatten() - y_true.flatten()

    fig, ax = plt.subplots(figsize=(12, 8), dpi=100)

    # Scatter plot of residuals
    ax.scatter(y_true, residuals,
               s=100,
               alpha=0.8,
               c=np.abs(residuals),
               cmap='coolwarm',
               edgecolor='white',
               linewidth=0.8)

    # Add a horizontal line at y=0
    ax.axhline(y=0, color='firebrick', linestyle='--', linewidth=2)

    # Add a trend line for residuals
    z = np.polyfit(y_true.flatten(), residuals, 1)
    p = np.poly1d(z)
    ax.plot(np.sort(y_true.flatten()),
            p(np.sort(y_true.flatten())),
            "k--",
            linewidth=1.5,
            label=f'Trend: y={z[0]:.4f}x+{z[1]:.4f}')

    # Styling
    ax.set_xlabel('Actual Values', fontweight='bold')
    ax.set_ylabel('Residuals (Predicted - Actual)', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=20)

    # Add statistics about residuals
    stats_text = (f"Mean Residual: {np.mean(residuals):.4f}\n"
                  f"Std Dev: {np.std(residuals):.4f}")

    # Add stats text box
    props = dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='gray')
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props)

    # Set grid style
    ax.grid(True, linestyle='--', alpha=0.5)

    # Add legend
    ax.legend(loc='lower right')

    # Tight layout
    plt.tight_layout()

    return fig

# Create and save all plots
scatter_fig = plot_scatter_comparison(y_test_inv, y_test_pred_inv, metrics,
                                     "PINN Model: Actual vs Predicted Propagation Distance")
scatter_fig.savefig('pinn_scatter_comparison.png', dpi=300, bbox_inches='tight')

line_fig = plot_line_comparison(y_test_inv, y_test_pred_inv,
                               "PINN Model: Comparison of Actual and Predicted Propagation Distance")
line_fig.savefig('pinn_line_comparison.png', dpi=300, bbox_inches='tight')

residual_fig = plot_residuals(y_test_inv, y_test_pred_inv,
                             "PINN Model: Residual Analysis")
residual_fig.savefig('pinn_residual_analysis.png', dpi=300, bbox_inches='tight')

# Display all plots
plt.show()

# 4. Feature importance analysis (new)
def plot_feature_importance(model, feature_names):
    # Create synthetic data spanning the range of each feature
    n_samples = 1000
    n_features = len(feature_names)
    synthetic_data = np.zeros((n_samples * n_features, n_features))

    # Set baseline values (means of original features)
    baseline = np.mean(X_train, axis=0)

    # For each feature, create variations while keeping others constant
    for i in range(n_features):
        feature_min = np.min(X_train[:, i])
        feature_max = np.max(X_train[:, i])
        feature_range = np.linspace(feature_min, feature_max, n_samples)

        # Create samples where only this feature varies
        for j in range(n_samples):
            synthetic_data[i*n_samples + j] = baseline.copy()
            synthetic_data[i*n_samples + j, i] = feature_range[j]

    # Convert to tensor
    synthetic_tensor = torch.tensor(synthetic_data, dtype=torch.float32)

    # Get predictions
    model.eval()
    with torch.no_grad():
        predictions = model(synthetic_tensor).numpy()

    # Calculate feature importance as the range of predictions when varying each feature
    importance = np.zeros(n_features)
    for i in range(n_features):
        feature_preds = predictions[i*n_samples:(i+1)*n_samples]
        importance[i] = np.max(feature_preds) - np.min(feature_preds)

    # Normalize importance
    importance = importance / np.sum(importance)

    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8), dpi=100)

    # Sort features by importance
    sorted_idx = np.argsort(importance)
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_importance = importance[sorted_idx]

    # Create barplot
    bars = ax.barh(sorted_features, sorted_importance,
            color=plt.cm.viridis(np.linspace(0, 0.8, len(sorted_features))),
            edgecolor='black', linewidth=1.2, alpha=0.8)

    # Add percentage labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        label = f"{width*100:.1f}%"
        ax.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                label, va='center', ha='left', fontweight='bold')

    # Styling
    ax.set_xlabel('Relative Importance', fontweight='bold')
    ax.set_ylabel('Features', fontweight='bold')
    ax.set_title('Feature Importance Analysis', fontweight='bold', pad=20)
    ax.set_xlim(0, max(importance) * 1.2)

    # Add gridlines
    ax.grid(True, linestyle='--', alpha=0.7, axis='x')

    # Tight layout
    plt.tight_layout()

    return fig

# 5. Prediction surface plot for pairs of features
def plot_prediction_surface(model, feature_x_idx, feature_y_idx, feature_names, scaler, y_scaler):
    # Create a grid of values for the two selected features
    n_points = 30

    # Get feature ranges from the training data
    x_min, x_max = np.min(X_train[:, feature_x_idx]), np.max(X_train[:, feature_x_idx])
    y_min, y_max = np.min(X_train[:, feature_y_idx]), np.max(X_train[:, feature_y_idx])

    # Create grid
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, n_points),
                         np.linspace(y_min, y_max, n_points))

    # Create input data for model
    grid_points = np.zeros((n_points * n_points, X_train.shape[1]))

    # Set all features to their mean values
    for i in range(X_train.shape[1]):
        grid_points[:, i] = np.mean(X_train[:, i])

    # Set the two selected features to the grid values
    grid_points[:, feature_x_idx] = xx.ravel()
    grid_points[:, feature_y_idx] = yy.ravel()

    # Convert to tensor
    grid_tensor = torch.tensor(grid_points, dtype=torch.float32)

    # Get predictions
    model.eval()
    with torch.no_grad():
        z_pred = model(grid_tensor).numpy()

    # Reshape for contour plot
    z_pred = z_pred.reshape(xx.shape)

    # Inverse transform to get actual values
    z_pred_actual = y_scaler.inverse_transform(z_pred)

    # Create figure
    fig = plt.figure(figsize=(14, 10), dpi=100)
    ax = fig.add_subplot(111, projection='3d')

    # Create surface plot
    surf = ax.plot_surface(xx, yy, z_pred_actual, cmap='viridis',
                          linewidth=0, antialiased=True, alpha=0.8)

    # Add colorbar
    cbar = fig.colorbar(surf, ax=ax, shrink=0.7, pad=0.1)
    cbar.set_label('Predicted Propagation Distance', rotation=270, labelpad=20, fontsize=12, fontweight='bold')

    # Add scatter points for training data to see the distribution
    train_points_x = X_train[:, feature_x_idx]
    train_points_y = X_train[:, feature_y_idx]
    train_points_z = y_train

    ax.scatter(train_points_x, train_points_y, train_points_z,
               color='red', s=50, alpha=0.6, label='Training Data')

    # Styling
    ax.set_xlabel(feature_names[feature_x_idx], fontweight='bold', labelpad=10)
    ax.set_ylabel(feature_names[feature_y_idx], fontweight='bold', labelpad=10)
    ax.set_zlabel('Propagation Distance', fontweight='bold', labelpad=10)

    title = f'PINN Prediction Surface: Effect of {feature_names[feature_x_idx]} and {feature_names[feature_y_idx]}'
    ax.set_title(title, fontweight='bold', pad=20)

    # Add legend
    ax.legend()

    # Rotate the axes for better visualization
    ax.view_init(elev=30, azim=135)

    # Tight layout
    plt.tight_layout()

    return fig

# Feature importance plot
feature_names = ['Pressure (bar)', 'Bulk Temperature (K)', 'Methane Concentration (mol/m³)', 'Sub-cooling (K)']
importance_fig = plot_feature_importance(model, feature_names)
importance_fig.savefig('pinn_feature_importance.png', dpi=300, bbox_inches='tight')

# Create prediction surfaces for important feature pairs
# 1. Pressure vs Temperature
surface_fig1 = plot_prediction_surface(model, 0, 1, feature_names, scaler, y_scaler)
surface_fig1.savefig('pinn_surface_pressure_temp.png', dpi=300, bbox_inches='tight')

# 2. Pressure vs Methane Concentration
surface_fig2 = plot_prediction_surface(model, 0, 2, feature_names, scaler, y_scaler)
surface_fig2.savefig('pinn_surface_pressure_methane.png', dpi=300, bbox_inches='tight')

# 3. Temperature vs Sub-cooling
surface_fig3 = plot_prediction_surface(model, 1, 3, feature_names, scaler, y_scaler)
surface_fig3.savefig('pinn_surface_temp_subcooling.png', dpi=300, bbox_inches='tight')

plt.show()

# 6. Create a dashboard-style summary of results
def create_results_dashboard(model, X_test_tensor, y_test_tensor, y_test_inv, y_test_pred_inv, metrics, feature_names):
    """Create a comprehensive dashboard of PINN model results"""

    # Set up the figure with a grid of subplots
    fig = plt.figure(figsize=(20, 16), dpi=100)

    # Create a grid layout
    gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

    # Add a title to the entire figure
    fig.suptitle('Physics-Informed Neural Network (PINN) Model Results',
                 fontsize=20, fontweight='bold', y=0.98)

    # 1. Scatter plot (Actual vs Predicted)
    ax1 = fig.add_subplot(gs[0, 0])
    errors = np.abs(y_test_inv.flatten() - y_test_pred_inv.flatten())
    scatter = ax1.scatter(y_test_inv, y_test_pred_inv, c=errors, cmap='viridis',
                         s=80, alpha=0.8, edgecolor='white', linewidth=0.5)

    # Add perfect prediction line
    line_min = min(y_test_inv.min(), y_test_pred_inv.min()) * 0.9
    line_max = max(y_test_inv.max(), y_test_pred_inv.max()) * 1.1
    ax1.plot([line_min, line_max], [line_min, line_max],
             color='red', linestyle='--', linewidth=2)

    ax1.set_xlabel('Actual Values', fontweight='bold')
    ax1.set_ylabel('Predicted Values', fontweight='bold')
    ax1.set_title('Actual vs Predicted Values', fontweight='bold')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. Line plot (Comparison over samples)
    ax2 = fig.add_subplot(gs[0, 1:])
    indices = np.arange(len(y_test_inv))
    ax2.plot(indices, y_test_inv, marker='o', markersize=6, linewidth=2,
             label='Actual', color=custom_palette[0], alpha=0.9)
    ax2.plot(indices, y_test_pred_inv, marker='x', markersize=8, linewidth=2,
             linestyle='--', label='Predicted', color=custom_palette[1], alpha=0.9)

    ax2.set_xlabel('Sample Index', fontweight='bold')
    ax2.set_ylabel('Propagation Distance', fontweight='bold')
    ax2.set_title('Comparison of Actual and Predicted Values', fontweight='bold')
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='upper right')

    # 3. Residual plot
    ax3 = fig.add_subplot(gs[1, 0])
    residuals = y_test_pred_inv.flatten() - y_test_inv.flatten()
    ax3.scatter(y_test_inv, residuals, s=80, alpha=0.8, c=np.abs(residuals),
               cmap='coolwarm', edgecolor='white', linewidth=0.5)
    ax3.axhline(y=0, color='red', linestyle='--', linewidth=2)

    ax3.set_xlabel('Actual Values', fontweight='bold')
    ax3.set_ylabel('Residuals', fontweight='bold')
    ax3.set_title('Residual Analysis', fontweight='bold')
    ax3.grid(True, linestyle='--', alpha=0.5)

    # 4. Feature Importance
    ax4 = fig.add_subplot(gs[1, 1:])

    # Calculate importance (simplified for dashboard)
    importance = np.array([0.4, 0.3, 0.2, 0.1])  # Replace with actual calculation

    # Sort features by importance
    sorted_idx = np.argsort(importance)
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_importance = importance[sorted_idx]

    # Create horizontal barplot
    bars = ax4.barh(sorted_features, sorted_importance,
                   color=plt.cm.viridis(np.linspace(0, 0.8, len(sorted_features))),
                   edgecolor='black', linewidth=1, alpha=0.8)

    # Add percentage labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        label = f"{width*100:.1f}%"
        ax4.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                label, va='center', ha='left', fontweight='bold')

    ax4.set_xlabel('Relative Importance', fontweight='bold')
    ax4.set_title('Feature Importance Analysis', fontweight='bold')
    ax4.grid(True, linestyle='--', alpha=0.7, axis='x')

    # 5. Error Distribution (histogram of residuals)
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.hist(residuals, bins=15, alpha=0.7, color=custom_palette[2], edgecolor='black')
    ax5.axvline(x=0, color='red', linestyle='--', linewidth=2)

    ax5.set_xlabel('Residual Value', fontweight='bold')
    ax5.set_ylabel('Frequency', fontweight='bold')
    ax5.set_title('Distribution of Residuals', fontweight='bold')
    ax5.grid(True, linestyle='--', alpha=0.5)

    # 6. Metrics Table
    ax6 = fig.add_subplot(gs[2, 1:])
    ax6.axis('tight')
    ax6.axis('off')

    # Create table data
    table_data = [
        ['Metric', 'Value'],
        ['Mean Squared Error (MSE)', f"{metrics['mse']:.4f}"],
        ['Mean Absolute Error (MAE)', f"{metrics['mae']:.4f}"],
        ['R-squared (R²)', f"{metrics['r2']:.4f}"],
        ['Mean Residual', f"{np.mean(residuals):.4f}"],
        ['Standard Deviation of Residuals', f"{np.std(residuals):.4f}"]
    ]

    # Create table
    table = ax6.table(cellText=table_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    # Style the table
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontproperties=dict(weight='bold'))
            cell.set_facecolor('#4c72b0')
            cell.set_text_props(color='white')
        elif col == 0:
            cell.set_text_props(fontproperties=dict(weight='bold'))

    ax6.set_title('Model Performance Metrics', fontweight='bold')

    # Add model architecture information at the bottom
    plt.figtext(0.5, 0.01,
                "PINN Architecture: 4 inputs → 256 → 128 → 64 → 32 → 1 output | Training: 8000 epochs with Adam optimizer",
                ha="center", fontsize=12, bbox={"boxstyle":"round", "facecolor":"white", "alpha":0.8, "pad":0.5})

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    return fig

# Create the dashboard
dashboard_fig = create_results_dashboard(model, X_test_tensor, y_test_tensor,
                                        y_test_inv, y_test_pred_inv, metrics, feature_names)
dashboard_fig.savefig('pinn_results_dashboard.png', dpi=300, bbox_inches='tight')

print("All visualization plots have been created and saved!")